# 5차시 (4) 멀티 에이전트 — 직접 코드로 만들기

한경국립대 2026 여름특강 · 딥러닝/머신러닝 입문 (초급반)

> **2차시(온라인)** 에서 우리는 프롬프트·RAG·에이전트를 **완성된 AI 서비스로 체험**했습니다.
> **오늘(오프라인)** 은 그걸 *서비스로 쓰는 게 아니라*, **직접 코드로 만들어 화면으로 확인**합니다.

앞 노트북에서 만든 챗봇/RAG는 "질문 하나 → 답 하나" 구조였습니다.
이번에는 한 걸음 더 나아가, **의도에 따라 갈라지고(라우팅) · 도구를 스스로 쓰는(tool use)** 에이전트를 코드로 만듭니다.

- **Part 1. 의도 분류 라우팅** — 질문의 *의도* 를 파악해 알맞은 처리로 보내기
- **Part 2. 도구 사용(tool use)** — 모델이 스스로 계산·검색 도구를 호출해 답하기
- **Part 3. 멀티 에이전트 & LangGraph** — 라우팅+도구를 엮는 큰 그림(개념)

> **에이전트(agent)** = 스스로 *계획하고, 도구를 쓰고, 반복* 하며 일을 해내는 AI. 오늘 그 핵심 두 조각(라우팅·도구 사용)을 손으로 만들어 봅니다.

## 0. 환경 세팅
라이브러리 설치 + API 키 입력 (앞 노트북과 동일).

> ⚠️ 이 노트북의 코드 셀은 대부분 **OpenAI API** 를 호출합니다. 반드시 **Colab 등 API 키가 설정된 환경에서 실행**하세요. (문법 검증용 로컬 환경에서는 실행되지 않습니다.)

In [ ]:
!pip install -qU langchain-openai langchain-core
print("설치 완료!")

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API 키를 입력하세요: ")

print("API 키 설정 완료" if os.environ.get("OPENAI_API_KEY") else "키가 비어 있습니다.")

---
# Part 1 · 의도 분류 라우팅 (Routing)

실제 챗봇 서비스는 들어온 질문을 곧장 LLM에 던지지 않고, **먼저 의도를 분류** 한 뒤 알맞은 곳으로 보냅니다.
예: "환불해줘" → 환불 담당 / "오늘 날씨" → 검색 도구 / "우리 회사 휴가 규정" → RAG.

이것을 **라우팅(routing)** 이라 하고, 의도를 가르는 부분이 **라우터(router)** 입니다.
가장 간단한 방법은 **LLM에게 분류를 시키는 것** 입니다.

먼저 의도를 분류하는 라우터를 만들어 봅시다.

In [ ]:
from langchain_openai import ChatOpenAI

router_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

INTENTS = ["날씨", "계산", "일반대화"]

def classify_intent(message):
    prompt = f"""다음 사용자의 말을 아래 의도 중 하나로만 분류해줘. 다른 말 없이 의도 단어만 출력해.

가능한 의도: {', '.join(INTENTS)}

사용자: {message}
의도:"""
    result = router_llm.invoke(prompt).content.strip()
    # 혹시 모를 군더더기 제거: 정의된 의도 중 매칭되는 것만 채택
    for it in INTENTS:
        if it in result:
            return it
    return "일반대화"

# 테스트
for msg in ["내일 서울 날씨 어때?", "123 곱하기 47은?", "심심한데 농담 하나 해줘"]:
    print(f"'{msg}'  ->  의도: {classify_intent(msg)}")

분류된 의도에 따라 **다른 함수(처리기)** 로 보내면 라우팅이 완성됩니다.
각 의도별 처리를 연결해 봅시다(날씨·계산은 일단 단순 처리, 일반대화는 LLM).

In [ ]:
chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def handle_weather(message):
    # (실제로는 날씨 API를 호출합니다. 여기서는 예시 응답)
    return "[날씨 처리기] 날씨 정보는 기상청 API에 연결하면 됩니다. (예: 내일 서울 맑음, 최고 28도)"

def handle_calc(message):
    return "[계산 처리기] 계산이 필요하군요. Part 2에서 '도구 사용'으로 실제 계산을 해봅니다."

def handle_chat(message):
    return "[일반대화] " + chat_llm.invoke(message).content

def route(message):
    intent = classify_intent(message)
    if intent == "날씨":
        return handle_weather(message)
    elif intent == "계산":
        return handle_calc(message)
    else:
        return handle_chat(message)

print(route("오늘 부산 날씨 알려줘"))
print(route("안녕! 오늘 기분 어때?"))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 1 · 내 문장은 어떤 의도로 분류될까 (온램프)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 위 <code>classify_intent</code> 는 이미 잘 동작합니다. 아래 칸(<code>___</code>)에 <b>여러분이 직접 만든 문장</b>을 넣어, 라우터가 어떤 의도로 분류하는지 관찰하세요.<br><b>힌트</b> &nbsp; 확실한 문장(날씨·계산·잡담)부터 넣어 보고, 그다음 애매한 문장(예: "이번 주말 우산 챙길까?")도 넣어 어떻게 분류되는지 보세요.<br><b>관찰 포인트</b> &nbsp; 세 문장이 각각 <code>[날씨]</code>·<code>[계산]</code>·<code>[일반대화]</code> 로 갈리면 성공. 애매한 문장은 예상과 다르게 분류될 수 있어요(그게 규칙 없는 LLM 라우터의 한계).
</div>
</div>

In [ ]:
# ⚠️ Colab에서 실행 (OpenAI API 필요)
my_messages = [
    "___",   # 날씨를 묻는 문장 (예: 내일 ~ 날씨)
    "___",   # 계산을 시키는 문장 (예: ~ 곱하기 ~)
    "___",   # 그냥 잡담 (예: 농담·추천·인사)
]
for m in my_messages:
    print(f"[{classify_intent(m)}] {m}")

> **연결 포인트**: 직전 노트북(BERT 분류)에서 *의도 데이터* 로 분류기를 학습했다면, 위 `classify_intent` 를 그 **BERT 분류기로 바꿔** 더 빠르고 저렴하게 라우팅할 수 있습니다(LLM 호출 비용↓).
> 또는 2~4차시의 **임베딩·유사도** 를 써서, 각 의도 예시 문장과의 유사도로 분류하는 **시맨틱 라우팅** 도 가능합니다.

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 2 · 라우터에 '번역' 의도 추가하기 (빈칸 채우기)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 라우터에 <b>'번역' 의도</b>를 추가해 "영어로 번역해줘" 같은 질문이 <b>번역 처리기</b>로 가게 만드세요. 아래 코드의 빈칸(<code>___</code>) 두 곳을 채우면 완성됩니다.<br><b>힌트</b> &nbsp; ① <code>INTENTS_V2</code> 의 <code>'___'</code> 자리에는 의도 이름을 <b>문자열</b>로 넣습니다. ② <code>handle_translate</code> 의 프롬프트는 <code>message</code> 를 넣어 <b>"설명 없이 자연스러운 영어로 번역만"</b> 하도록 f-string(<code>f"...{message}..."</code>)으로 지시하세요.<br><b>예상</b> &nbsp; "이 문장 영어로 번역해줘: 나는 학생입니다" → <code>[번역] I am a student.</code> 처럼 갈립니다. 채우기 전엔 번역이 라우팅되지 않아요.
</div>
</div>

In [ ]:
# ⚠️ Colab에서 실행 (OpenAI API 필요)
# 1) 의도 목록에 '번역' 추가
INTENTS_V2 = ["날씨", "계산", "___", "일반대화"]   # 빈칸('___')을 의도 이름 문자열로 채우세요

def classify_intent_v2(message):
    prompt = f"""다음 사용자의 말을 아래 의도 중 하나로만 분류해줘. 의도 단어만 출력해.
가능한 의도: {', '.join(INTENTS_V2)}
사용자: {message}
의도:"""
    result = router_llm.invoke(prompt).content.strip()
    for it in INTENTS_V2:
        if it in result:
            return it
    return "일반대화"

# 2) 번역 처리기 — 프롬프트를 직접 완성하세요
def handle_translate(message):
    prompt = f"___"   # message 를 '설명 없이 자연스러운 영어로 번역만' 하도록 지시하는 f-string 을 쓰세요
    return "[번역] " + chat_llm.invoke(prompt).content

# 3) 라우팅에 번역 분기 추가 (완성됨)
def route_v2(message):
    intent = classify_intent_v2(message)
    if intent == "번역":
        return handle_translate(message)
    elif intent == "날씨":
        return handle_weather(message)
    elif intent == "계산":
        return handle_calc(message)
    else:
        return handle_chat(message)

# 4) 테스트 — 빈칸을 잘 채웠다면 번역/날씨/계산/잡담이 각각 알맞게 갈라집니다.
for msg in ["이 문장 영어로 번역해줘: 나는 학생입니다", "내일 서울 날씨 어때?", "3 더하기 4는?", "안녕! 반가워"]:
    print(f"[{classify_intent_v2(msg)}] {msg}")
    print("  ->", route_v2(msg))

---
# Part 2 · 도구 사용 (Tool Use / 함수 호출)

라우터로 의도를 *우리가* 나눠 줄 수도 있지만, 똑똑한 모델은 **스스로 필요한 도구를 골라** 쓰기도 합니다.
이를 **도구 사용(tool use)** 또는 **함수 호출(function calling)** 이라 합니다.

흐름:
1. 우리가 모델에게 "이런 도구들이 있어"라고 **도구 목록(함수)** 을 알려줍니다.
2. 질문이 들어오면 모델이 **"이 도구를 이런 인자로 써!"** 라고 알려줍니다.
3. 우리가 그 도구(함수)를 실제로 **실행** 하고, 결과를 모델에게 돌려줍니다.
4. 모델이 그 결과로 **최종 답** 을 만듭니다.

먼저 모델이 쓸 **도구(함수)** 두 개를 정의합니다: 계산기와 (가짜)검색.

In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """수식을 계산한다. 예: '12 * (3 + 4)'. 사칙연산과 괄호를 지원한다."""
    try:
        # 안전을 위해 숫자와 기본 연산자만 허용
        allowed = set("0123456789+-*/(). ")
        if not set(expression) <= allowed:
            return "허용되지 않은 문자가 있습니다."
        return str(eval(expression))
    except Exception as e:
        return f"계산 오류: {e}"

@tool
def web_search(query: str) -> str:
    """인터넷에서 정보를 검색한다. 최신 정보나 사실 확인이 필요할 때 사용한다."""
    # 실제로는 검색 API(예: Tavily, SerpAPI)를 호출합니다.
    # 이번 실습에서는 키 없이 동작하도록 '가짜 검색 결과'를 돌려줍니다.
    fake_db = {
        "한경국립대": "한경국립대학교는 경기도 안성/평택에 캠퍼스를 둔 국립대학교입니다.",
        "파이썬": "파이썬(Python)은 1991년 귀도 반 로섬이 발표한 프로그래밍 언어입니다.",
    }
    for key, val in fake_db.items():
        if key in query:
            return val
    return "(가짜 검색) 관련 결과를 찾지 못했습니다. 실제 서비스에서는 검색 API를 연결하세요."

print("도구 2개 준비 완료: calculator, web_search")

이제 모델에 도구를 **연결(bind)** 하고, 모델이 도구를 호출하면 실제로 실행해 결과를 돌려주는 **루프** 를 만듭니다.
이 루프가 바로 '에이전트' 의 핵심 동작입니다: *모델이 도구를 부르면 → 실행 → 결과를 다시 모델에게 → 최종 답*.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage

tools = [calculator, web_search]
tools_by_name = {t.name: t for t in tools}

# 모델에게 '이런 도구를 쓸 수 있다'고 알려줌
agent_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

def run_agent(question):
    messages = [HumanMessage(content=question)]

    # 1) 모델에게 질문 -> 모델이 '도구를 쓰자'고 할 수 있음
    ai_msg = agent_llm.invoke(messages)
    messages.append(ai_msg)

    # 2) 모델이 요청한 도구가 있으면 실제로 실행
    if ai_msg.tool_calls:
        for call in ai_msg.tool_calls:
            tool = tools_by_name[call["name"]]
            result = tool.invoke(call["args"])
            print(f"  [도구 실행] {call['name']}({call['args']}) -> {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
        # 3) 도구 결과를 모델에게 다시 줘서 최종 답 생성
        final = agent_llm.invoke(messages)
        return final.content
    else:
        # 도구가 필요 없으면 바로 답
        return ai_msg.content

print("질문: 137 곱하기 256은 얼마야?")
print("답:", run_agent("137 곱하기 256은 얼마야?"))

In [ ]:
# 검색 도구가 필요한 질문
print("질문: 한경국립대가 어떤 학교야?")
print("답:", run_agent("한경국립대가 어떤 학교야?"))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 3 · 모델이 어떤 도구를 고르는지 관찰 (관찰형)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 아래 세 질문을 실행해, 모델이 <b>어떤 도구를 고르는지</b>(또는 도구 없이 답하는지) <code>[도구 실행]</code> 로그로 확인하세요.<br><b>힌트</b> &nbsp; 계산이 필요한 질문 → <code>calculator</code>, 사실·검색이 필요한 질문 → <code>web_search</code>, 감정·잡담 → 도구 없이 답. 각 질문이 왜 그 도구를 골랐는지 도구의 <b>설명(docstring)</b>과 연결해 생각해 보세요.<br><b>관찰 포인트</b> &nbsp; 첫 질문엔 <code>calculator</code>, 둘째엔 <code>web_search</code> 로그가 찍히고, 셋째엔 로그 없이 대화로 답하면 성공.
</div>
</div>

In [ ]:
# ⚠️ Colab에서 실행 (OpenAI API 필요)
# 세 질문을 각각 실행해 도구 선택을 관찰하세요.
test_questions = [
    "123 곱하기 45는 얼마야?",     # → 계산기(calculator) 도구를 골라야 함
    "한경국립대는 어떤 학교야?",    # → 검색(web_search) 도구를 골라야 함
    "오늘 기분이 좀 우울해",        # → 도구 없이 그냥 대화로 답
]
for q in test_questions:
    print("질문:", q)
    print("답:", run_agent(q))
    print("-" * 45)

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 4 · 나만의 도구 만들어 붙이기 (직접 작성)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 모델이 스스로 고를 <b>새 도구</b>를 하나 직접 정의해 에이전트에 붙이고, 그 도구가 필요한 질문을 던져 <code>[도구 실행]</code> 로그가 찍히는지 확인하세요.<br><b>힌트</b> &nbsp; <code>@tool</code> 아래 <b>docstring(설명)</b>에 "이 도구를 언제 쓰는지"를 한 문장으로 명확히 쓰세요(모델은 이 설명을 읽고 고릅니다). 동작은 파이썬으로 간단히 — 아이디어: 글자 수 세기(<code>len(text)</code>) · 문자열 뒤집기(<code>text[::-1]</code>) · 섭씨↔화씨 변환 등. 마지막 줄엔 그 도구가 필요한 질문을 넣으세요.<br><b>예상</b> &nbsp; 예를 들어 '글자 수 세기' 도구라면 "'안녕하세요'는 몇 글자야?" 질문에 <code>[도구 실행] my_tool(...)</code> 로그가 찍히고 답이 나옵니다.
</div>
</div>

In [ ]:
# ⚠️ Colab에서 실행 (OpenAI API 필요)
from langchain_core.tools import tool

# TODO: 아래 도구를 완성하세요. (아이디어: 글자 수 세기 · 랜덤 추천 · 온도 변환 등)
@tool
def my_tool(text: str) -> str:
    """___"""                       # 이 도구를 언제 쓰는지 한 문장으로 설명 (모델이 이걸 보고 고릅니다)
    # 실제 동작을 파이썬으로 구현하고 결과 문자열을 return 하세요
    return "___"

# 새 도구를 에이전트에 추가해 다시 묶기
tools_v2 = [calculator, web_search, my_tool]
tools_by_name.update({t.name: t for t in tools_v2})
agent_llm2 = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools_v2)

def run_agent2(question):
    messages = [HumanMessage(content=question)]
    ai_msg = agent_llm2.invoke(messages); messages.append(ai_msg)
    if ai_msg.tool_calls:
        for call in ai_msg.tool_calls:
            result = tools_by_name[call["name"]].invoke(call["args"])
            print(f"  [도구 실행] {call['name']}({call['args']}) -> {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
        return agent_llm2.invoke(messages).content
    return ai_msg.content

# my_tool 이 필요한 질문을 직접 던져 보세요.
print(run_agent2("___"))

> **실제 인터넷 검색을 붙이려면**: 위 `web_search` 의 가짜 부분을 실제 검색 API로 바꾸면 됩니다.
> 대표적으로 **Tavily**(`pip install langchain-tavily`, 무료 키 발급)나 SerpAPI 등을 연결합니다. 키가 필요하므로 이번 수업에서는 가짜 검색으로 흐름만 익혔습니다.

---
# Part 3 · (개념) 멀티 에이전트와 LangGraph

지금까지 본 **라우팅** 과 **도구 사용** 을 여러 개 엮으면 **멀티 에이전트 시스템** 이 됩니다.
예: *질문 의도 분류 → (검색 에이전트 / RAG 에이전트 / 계산 에이전트) → 결과를 정리하는 에이전트*.

이런 흐름을 **그래프(graph)** 로 그려 관리하는 대표 도구가 **LangGraph** 입니다.
- **노드(node)**: 하나의 작업(에이전트/함수). 예: '의도 분류', '검색', '답 생성'.
- **엣지(edge)**: 다음에 어디로 갈지(조건 분기 포함).
- **상태(state)**: 노드들이 공유하는 정보(대화·중간 결과).

오늘 우리가 손으로 만든 `route()` 함수가 바로 작은 LangGraph 한 장입니다.

```
           ┌──────────────┐
  질문 ──▶  │  의도 분류    │
           └──────┬───────┘
        ┌─────────┼─────────┐
        ▼         ▼         ▼
   [날씨 검색] [계산 도구] [RAG/대화]
        └─────────┼─────────┘
                  ▼
             최종 답 정리
```

**언제 무엇을 쓰나(2026 기준, 개념만)**
- 복잡하고 분기·상태가 많은 워크플로 → **LangGraph**
- 빠른 프로토타입, 역할 기반 팀 구성 → **CrewAI**
- RAG 중심 검색 에이전트 → **LlamaIndex**

> 초급 단계에서는 "라우팅 + 도구 사용" 개념만 확실히 잡으면 충분합니다. 프레임워크는 필요해질 때 골라 쓰면 됩니다.

---
## 마무리

오늘 (4)번 노트북에서 **직접 코드로 만든 것**:
- **라우팅**: 의도를 먼저 분류해 알맞은 처리로 보내기(LLM 라우터 → BERT·임베딩으로 대체 가능).
- **도구 사용**: 모델이 스스로 계산기·검색 도구를 호출 → 실행 → 결과로 최종 답.
- **멀티 에이전트 & LangGraph**(개념): 라우팅+도구를 여러 개 엮어 협업하는 큰 그림.

> **2차시에서 서비스로 봤던 것**(챗봇·RAG·에이전트·NotebookLM)을, 오늘은 **부품 단위로 직접 코드로** 만들어 봤습니다.

**5차시 전체 정리**: 프롬프트 엔지니어링 → 나만의 챗봇 → RAG 챗봇 → BERT 분류 → 멀티 에이전트.
작은 부품(프롬프트·임베딩·검색·분류·도구)을 조합하면 점점 똑똑한 AI 서비스가 됩니다. 수고하셨습니다!